# Visualisation & Filtering

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2
import hippo
import hippo.pset as pset
import mrich
from mrich import print
from pathlib import Path
from os import environ
import pandas as pd
import plotly.express as px
import molparse as mp
pset.mp = mp  

## Configuration

In [ ]:
target_name = "Target name"  # Change this to match the target name in Fragalysis.
cycle_number = 1  # Change this to match the design cycle you are scoring.
pose_tag = "gnina_repose" # tag applied to the poses you want to visualise; this should match the pose tag used in previous steps.
method_tags = ["fragmenstein", "pure_knitwork", "impure_knitwork"] # Select the elaboration methods you want to analyse. These should match the method tags used in previous steps.
score_col = "energy_score"  # column in score_df to rank poses by; also available: "distance_score", "cnn_vs"
score_quantile = 0.2  # keep the best-scoring 20% of poses for each method
selection_tag = f"c{cycle_number:02}_selected"  # tag applied to the shortlisted poses

In [ ]:
# all XChem-FFF work is kept under $HOME2/XChem-FFF; each target gets its own
# subdirectory here, and each design cycle its own subdirectory within that
xchem_fff_dir = Path(environ["HOME2"]) / "XChem-FFF"
target_dir = xchem_fff_dir / target_name.lower()
cycle_dir = target_dir / f"cycle_{cycle_number:02}"
gnina_dir = cycle_dir / "gnina"
gnina_outputs_dir = gnina_dir / "outputs"
bulk_targets_dir = Path(environ["BULK"]) / "TARGETS"
bulk_target_dir = bulk_targets_dir / target_name

## Animal

In [ ]:
animal = hippo.HIPPO(target_name, bulk_target_dir / f"{target_name}.sqlite")

In [ ]:
animal.tags  # Display available tags

## 1. Build a score table for the GNINA-minimised poses

In [ ]:
gnina_poses = animal.poses.get_by_tag(pose_tag)

def get_method(pose):
    # the method tag is only recorded on the pre-GNINA placed pose, not the GNINA-reposed
    # one, but load_sdf preserves the placed pose's tags in metadata["tags"]
    pose_tags = (pose.metadata.get("tags") or "").split(",")
    for method in method_tags:
        if f"{target_name}_{method}" in pose_tags:
            return method
    return None

data = []
for pose in gnina_poses:
    data.append({
        "pose_id": pose.id,
        "compound_id": pose.compound_id,
        "method": get_method(pose),
        "energy_score": pose.energy_score,
        "distance_score": pose.distance_score,
        # CNN_VS isn't a dedicated HIPPO score field, but GNINA's rescoring SDF columns
        # are preserved on each pose's metadata, so it can still be pulled out for triage.
        "cnn_vs": pose.metadata.get("CNN_VS"),
    })

score_df = pd.DataFrame(data).set_index("pose_id")
score_df.head()

In [ ]:
score_df.to_csv(gnina_dir / f"{target_name}_c{cycle_number:02}_scores.csv")

## 2. Visualise the score distributions

In [ ]:
px.histogram(score_df, x="energy_score", color="method", barmode="overlay")

In [ ]:
px.scatter(score_df, x="distance_score", y="energy_score", color="method", hover_data=["compound_id"])

In [ ]:
px.histogram(score_df, x="cnn_vs", color="method", barmode="overlay")

## 3. Select the top-scoring poses

In [ ]:
higher_is_better = {"cnn_vs"}  # cnn_vs favours larger values; energy_score/distance_score favour smaller ones

if score_col in higher_is_better:
    cutoffs = score_df.groupby("method")[score_col].quantile(1 - score_quantile)
    top_score_df = score_df[
        score_df.apply(lambda row: row[score_col] >= cutoffs[row["method"]], axis=1)
    ]
else:
    cutoffs = score_df.groupby("method")[score_col].quantile(score_quantile)
    top_score_df = score_df[
        score_df.apply(lambda row: row[score_col] <= cutoffs[row["method"]], axis=1)
    ]

top_score_df

In [ ]:
top_poses = animal.poses[list(top_score_df.index)]
top_poses

## 4. Generate pose overlays and images of the top and bottom poses

In [ ]:
for pose in top_poses[:5]:
    print(pose)
    pose.draw()  # 2D depiction of the pose's compound
for pose in top_poses[-5:]:
    print(pose)
    pose.draw()  # 2D depiction of the pose's compound

In [ ]:
# eyeball each top-scoring pose alongside its inspirations to sanity-check
# the placement/merge before committing to the selection
for pose in top_poses[:5]:
    inspirations = pose.inspirations
    print(pose, inspirations.names)
    (pose + inspirations).draw()
for pose in top_poses[-5:]:
    inspirations = pose.inspirations
    print(pose, inspirations.names)
    (pose + inspirations).draw()

## 4b. Interactively accept/reject poses

Step through each top-scoring pose one at a time and decide whether to keep it in the final
selection. Your decisions are stored in `decisions` (`pose_id -> True/False`); re-run the cell
below to start over, or just re-run the review cell if you change your mind on any pose.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

decisions = {}  # pose_id -> True (accept) / False (reject)
review_poses = list(top_poses)
index = {"i": 0}

output = widgets.Output()
accept_btn = widgets.Button(description="Accept", button_style="success")
reject_btn = widgets.Button(description="Reject", button_style="danger")
progress = widgets.Label()

def show_pose():
    output.clear_output(wait=True)
    progress.value = f"Pose {index['i'] + 1} / {len(review_poses)}"
    if index["i"] >= len(review_poses):
        with output:
            print("All poses reviewed.")
        accept_btn.disabled = True
        reject_btn.disabled = True
        return
    pose = review_poses[index["i"]]
    with output:
        inspirations = pose.inspirations
        print(pose, inspirations.names)
        (pose + inspirations).draw()

def record_decision(accept):
    pose = review_poses[index["i"]]
    decisions[pose.id] = accept
    index["i"] += 1
    show_pose()

accept_btn.on_click(lambda b: record_decision(True))
reject_btn.on_click(lambda b: record_decision(False))

display(widgets.HBox([accept_btn, reject_btn]), progress, output)
show_pose()

In [ ]:
# once you've been through all poses, build the accepted subset
accepted_ids = [pose_id for pose_id, accept in decisions.items() if accept]
missing = [pose.id for pose in review_poses if pose.id not in decisions]
if missing:
    print(f"Warning: {len(missing)} pose(s) not yet reviewed: {missing}")

selected_poses = animal.poses[accepted_ids]
selected_poses

## 5. Tag the selected poses

Attribute the `selection_tag` (defined in the config section above) to the shortlisted poses.
use it later to re-run scoring/placement/docking on the same poses, to export an SDF for other tools, or to pull out SMILES for compound ordering / recipe generation in CAR — via `animal.poses.get_by_tag(<selection_tag>)` 

In [ ]:
selected_poses.add_tag(selection_tag)
selection_tag

## 6. Export the tagged poses

In [ ]:
top_score_df.loc[accepted_ids].to_csv(gnina_dir / f"{target_name}_c{cycle_number:02}_top_scoring_poses.csv")

In [ ]:
selected_poses.to_fragalysis(
    str(gnina_dir / f"{target_name}_c{cycle_number:02}_top_scoring_poses.sdf"),
    method = "gnina",
    submitter_name = "YOUR NAME HERE",
    submitter_email = "YOUR EMAIL HERE",
    submitter_institution = "YOUR INSTITUTION HERE",
    copy_reference_pdbs=True
)

In [ ]:
# Export the SMILES of the shortlisted compounds, e.g. for compound ordering or recipe generation in CAR.
# CompoundSet.smiles/.ids return plain lists, so build a DataFrame before writing the CSV.

pd.DataFrame({
    "compound_id": selected_poses.compounds.ids,
    "smiles": selected_poses.compounds.smiles,
}).to_csv(
    gnina_dir / f"{target_name}_c{cycle_number:02}_top_scoring_compounds_smiles.csv",
    index=False,
)

In [ ]:
animal.db.backup()  # save the updated HIPPO database, preserving scores and tags